In [1]:
import os
import random
import shutil
from pathlib import Path

# 1. Configuration - Define BOTH source folders
DATA_ROOT = Path("/home/wiebe/Desktop/Robotic/AE4317/testing_nn/data")
IMAGE_SOURCE = DATA_ROOT / "pre-procest"
LABEL_SOURCE = DATA_ROOT / "annotations"  
EXPORT_DIR = DATA_ROOT / "dataset"

def split_dataset(split_ratio=0.8):
    # 2. Create YOLO structure
    for folder in ['train', 'val']:
        (EXPORT_DIR / folder / 'images').mkdir(parents=True, exist_ok=True)
        (EXPORT_DIR / folder / 'labels').mkdir(parents=True, exist_ok=True)

    # 3. Identify only the annotated pairs by looking in the LABEL_SOURCE
    label_files = [f for f in os.listdir(LABEL_SOURCE) if f.endswith('.txt') and f != 'classes.txt']
    
    valid_pairs = []
    for lbl in label_files:
        base_name = Path(lbl).stem
        # Check if the matching image exists in the IMAGE_SOURCE folder
        for ext in ['.jpg', '.png', '.jpeg']:
            img_name = base_name + ext
            if (IMAGE_SOURCE / img_name).exists():
                valid_pairs.append((img_name, lbl))
                break

    if not valid_pairs:
        print(f"Error: No matching pairs found between {IMAGE_SOURCE} and {LABEL_SOURCE}")
        return

    # 4. Shuffle and Split
    random.shuffle(valid_pairs)
    split_idx = int(len(valid_pairs) * split_ratio)
    train_pairs = valid_pairs[:split_idx]
    val_pairs = valid_pairs[split_idx:]

    # 5. Helper to copy from two different sources
    def move_files(pair_list, subset):
        for img_name, lbl_name in pair_list:
            # Copy Image from IMAGE_SOURCE
            shutil.copy(IMAGE_SOURCE / img_name, EXPORT_DIR / subset / 'images' / img_name)
            # Copy Label from LABEL_SOURCE
            shutil.copy(LABEL_SOURCE / lbl_name, EXPORT_DIR / subset / 'labels' / lbl_name)

    move_files(train_pairs, 'train')
    move_files(val_pairs, 'val')
    
    print(f"Split complete: {len(train_pairs)} train, {len(val_pairs)} val.")

if __name__ == "__main__":
    split_dataset()

Split complete: 72 train, 19 val.
